# Time Domain: Driven Trapping Square

This example evolves the scalar wave equation on the compactified trapping geometry. The spatial weak form is the quadratic pencil $k^2M+ikC+K_{\rm Helmholtz}$. With time dependence $e^{-ikt}$, the corresponding semidiscretization is

$$
M\ddot q+C\dot q+Kq=f(t),\qquad K=-K_{\rm Helmholtz}.
$$

The matrix $C$ contains the hyperboloidal transport terms and the contribution at null infinity; no absorbing boundary condition is added. We use average-acceleration Newmark with $\beta=1/4$ and $\gamma=1/2$. The acceleration solve uses $S^{-1}=(M+\gamma\Delta t C+\beta\Delta t^2K)^{-1}$, so the assembled load is acted on by this same effective inverse. A localized harmonic source supported in the unchanged physical region is switched off before the final time to leave a short ringdown interval. The animation stores one frame every $0.5$ time units plus the final state.

In [ ]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw
import numpy as np
import matplotlib.pyplot as plt
import time
notebook_start = time.perf_counter()

In [ ]:
# domain parameters
Rout = 2
RScat = 1
DScat = 0.1
DHole = 0.2
R=1.5

# Discretization parameters
maxh = 0.1  
order = 4

# Geometry and forcing parameters
xOffset = 0.5
drive_frequency = 10.8
drive_until = 6.0
t_end = 10.0

In [ ]:
def create_geo(Rout = Rout, RScat = RScat, DScat = DScat, DHole = DHole, R = R, xOffset = xOffset,draw = True):
    domain = Circle((0, 0), Rout).Face()
    domain.edges.name = 'outer'
    domain.faces.name = 'outer'
    
    inner = Circle((0, 0), R).Face()
    inner.edges.name = 'interface'
    inner.faces.name = 'inner'

    scatterer = (
            MoveTo(-RScat/2+xOffset,-RScat/2).Rectangle(RScat,RScat).Face()
            -MoveTo(-RScat/2+DScat/2+xOffset,-RScat/2+DScat/2).Rectangle(RScat-DScat,RScat-DScat).Face()
            -MoveTo(-RScat/2-DScat+xOffset,-DHole/2).Rectangle(2*DScat,DHole).Face()
        )
    scatterer.edges.name = 'scatterer'

    scattererdom = MoveTo(-RScat/2+xOffset,-RScat/2).Rectangle(RScat,RScat).Face()-scatterer
    scattererdom.faces.name = 'cavity'
    


    if draw:
        Draw(Glue([domain-inner,inner-scattererdom-scatterer, scattererdom-scatterer]))
    geo = OCCGeometry(Glue([domain-inner,inner-scattererdom-scatterer, scattererdom]), dim=2)
    return geo

In [ ]:
geo = create_geo(draw = False)
mesh = Mesh(geo.GenerateMesh(maxh=maxh))
mesh.Curve(order)
Draw(mesh)
print(mesh.GetMaterials(),mesh.GetBoundaries())

In [ ]:
def get_bilinear_forms(maxh = maxh, order = order):
    geo = create_geo(Rout = Rout, RScat = RScat, DScat = DScat, DHole = DHole, R = R, draw = False)
    mesh = Mesh(geo.GenerateMesh(maxh=maxh))
    mesh.Curve(order)

    fes = H1(mesh,order = order, dirichlet = "scatterer")

    w,w_ = fes.TnT()

    rho = sqrt(x**2+y**2)

    #Omega = CF([(Rout-rho)/(Rout-R),1,1])
    Omega = mesh.MaterialCF({"outer": (Rout-rho)/(Rout-R)},default=1)

    DOmega = Omega.Diff(rho)
    L = Omega-rho*DOmega
    G = Omega**2/L

    #H = CF([1 - G,0,0]) #characteristic preserving
    H = mesh.MaterialCF({"outer": 1-G},default=0) #characteristic preserving
    Mu = 1+H  # (1-H**2)/G = 1+H when H=1-G


    vx = CF((x,y))

    Pi = OuterProduct(vx,vx)/rho**2 #projection onto radial vector
    Pi_perp = Id(2)-Pi              #projection onto tangential vector

    ### k**2
    M_ = Mu*w*w_*dx

    ### 1j*k
    C_ = (
        -H/rho*w*InnerProduct(vx,grad(w_))*dx
        +H/rho*w_*InnerProduct(vx,grad(w))*dx
        +w_*w*ds('outer')
        )

    ## 1
    K_ = (
        -grad(w)*( (G*Pi+L*Pi_perp)*grad(w_))*dx
        -Omega/L/2/rho*DOmega*InnerProduct(grad(w),vx)*w_*dx
        -Omega/L/2/rho*DOmega*InnerProduct(grad(w_),vx)*w*dx
        -1/L/4*DOmega**2*w*w_*dx
        )


    return M_, C_, K_,fes

In [ ]:
def compute_energy_norms(gfu,gfup):
    u,v = gfu.space.TnT()
    mass_scat = BilinearForm(u*v*dx('cavity'),check_unused=False).Assemble().mat
    mass_inner = BilinearForm(u*v*dx('inner|cavity'),check_unused=False).Assemble().mat
    
    stiffness_scat = BilinearForm(grad(u)*grad(v)*dx('cavity'),check_unused=False).Assemble().mat
    stiffness_inner = BilinearForm(grad(u)*grad(v)*dx('inner|cavity'),check_unused=False).Assemble().mat
    
    norm_inner = sqrt(gfup.vec.InnerProduct(mass_inner*gfup.vec)+gfu.vec.InnerProduct(stiffness_inner*gfu.vec)).real
    norm_scat = sqrt(gfup.vec.InnerProduct(mass_scat*gfup.vec)+gfu.vec.InnerProduct(stiffness_scat*gfu.vec)).real

    return norm_inner, norm_scat 

In [ ]:
def get_newmark_beta_matrices(M_,C_,K_,dt,gamma,beta,freedofs):
    M = BilinearForm(M_).Assemble().mat
    C = BilinearForm(C_).Assemble().mat
    K = BilinearForm((-1)*K_).Assemble().mat
    Sinv = BilinearForm(M_+gamma*dt*C_-beta*dt**2*K_).Assemble().mat.Inverse(freedofs = freedofs)
    Minv = M.Inverse(freedofs = freedofs)
    return M,C,K,Sinv,Minv

In [ ]:
dt = 2e-2

beta = 1/4
gamma = 1/2
assembly_start = time.perf_counter()
M_, C_, K_, fes = get_bilinear_forms(maxh = maxh, order = order)
M, C, K, Sinv, Minv = get_newmark_beta_matrices(M_,C_,K_,dt,gamma,beta,fes.FreeDofs())
assembly_seconds = time.perf_counter() - assembly_start
print(f"matrix assembly and factorization: {assembly_seconds:.3f} s")

In [ ]:
def run_simulation(omega, drive_until=drive_until, tend=t_end):
    w0 = exp(-((x+0.8)**2+(y+0.8)**2)*30)
        
    
    forcefact=100
    force = LinearForm(
        forcefact*w0*fes.TestFunction()*dx("inner|cavity")
    ).Assemble().vec
    effective_force = Sinv * force
    
    gfw = GridFunction(fes)
    #gfw.Set(w0)
    gfw.Set(0.)
    
    gfw_ = GridFunction(fes)
    gfw_.Set(0.)
    
    gfw__ = GridFunction(fes)
    gfw__.Set(0.)
    
    gfw__.vec.data = -Minv@K*gfw.vec-Minv@C*gfw_.vec
    
    tmp = gfw__.vec.CreateVector()
    
    gf_anim = GridFunction(fes,multidim=0)

    drawtimes = max(1, int(round(0.5/dt)))
    sample_times = []
    physical_norms = []
    cavity_norms = []
    
    # Average-acceleration Newmark method
    nsteps = int(round(tend/dt))
    for i in range(nsteps):
        t = i*dt
        if not i%drawtimes:
            norm_inner,norm_scat = compute_energy_norms(gfw,gfw_)
            sample_times.append(t)
            physical_norms.append(norm_inner)
            cavity_norms.append(norm_scat)
            print(f"t={t:5.2f}, physical norm={norm_inner:.4e}, cavity norm={norm_scat:.4e}",end='\r')
            gf_anim.AddMultiDimComponent(gfw.vec)
        tmp.data = gfw__.vec
        gfw__.vec.data = -Sinv @ K*gfw.vec-Sinv@C*gfw_.vec-dt*Sinv@K*gfw_.vec+(gamma-1)*dt*Sinv@C*tmp+(beta-1/2)*dt**2*Sinv@K*tmp+(t+dt<drive_until)*sin(omega*(t+dt))*effective_force
        gfw.vec.data += dt*gfw_.vec+(1/2-beta)*dt**2*tmp+beta*dt**2*gfw__.vec
        gfw_.vec.data += (1-gamma)*dt*tmp+gamma*dt*gfw__.vec

    norm_inner,norm_scat = compute_energy_norms(gfw,gfw_)
    sample_times.append(tend)
    physical_norms.append(norm_inner)
    cavity_norms.append(norm_scat)
    gf_anim.AddMultiDimComponent(gfw.vec)
    print(f"t={tend:5.2f}, physical norm={norm_inner:.4e}, cavity norm={norm_scat:.4e}",end='\r')
    print("")
    print("simulation done")
    print("")
    return gf_anim, np.array(sample_times), np.array(physical_norms), np.array(cavity_norms)


In [ ]:
evolution_start = time.perf_counter()
gf_anim, times, physical_norms, cavity_norms = run_simulation(drive_frequency)
evolution_seconds = time.perf_counter() - evolution_start
print(f"Newmark evolution ({int(round(t_end/dt))} steps): {evolution_seconds:.3f} s")

In [ ]:
Draw(gf_anim,min=-0.1,max=0.1,animate=True)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(times, physical_norms, label="physical region")
ax.semilogy(times, cavity_norms, label="cavity")
ax.axvline(drive_until, color="0.4", ls="--", label="source switched off")
ax.set(xlabel="time", ylabel="energy norm")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.show()
print(f"notebook compute time after imports: {time.perf_counter() - notebook_start:.3f} s")

## Interpretation

The animation shows the compactified time-domain unknown, which agrees with the physical field inside the interface and remains regular at null infinity. The norm history separates the full unchanged physical region from the cavity. After the source is switched off, energy can leave through the compactified exterior while a resonant component persists in the cavity.

This is a qualitative demonstration, not yet a convergence or long-time stability study. Such a study should vary mesh size, polynomial degree, and $\Delta t$, compare against a bounded-domain or PML reference in the unchanged physical region, and extend the ringdown interval.